In [ ]:
# import packages
import pandas as pd
from scipy.stats import pearsonr
import io

In [ ]:
# for reading the vcf
def read_vcf(path):
    with open(path, 'r') as f:
        lines = [l for l in f if not l.startswith('##')]
    return pd.read_csv(
        io.StringIO(''.join(lines)),
        sep='\t'
    )

In [ ]:
# calculate vaf for rna-seq and wgs; calculate the sum of VAF for each gene

def get_vaf(sample):
    file = 'results/hc-rna_hc-wgs/'+ sample + '_rna_wgs/'
    rna_shared = read_vcf(file + "/0002.vcf")
    wgs_shared = read_vcf(file + "/0003.vcf")

    # calculate VAF for each RNA-seq variant
    rna_shared[['allele', 'total']] = rna_shared[rna_shared.columns[9]].str.split(':', expand=True).iloc[:, [1, 2]]
    rna_shared[['ref', 'alt']] = rna_shared['allele'].str.split(',', expand = True).iloc[:, [0, 1]]
    rna_shared[['ref', 'alt', 'total']] = rna_shared[['ref', 'alt', 'total']].astype(int)
    rna_shared['vaf'] = rna_shared['alt']/rna_shared['total']
    rna_shared['type'] = "rna_shared"

    # calculate VAF for each WGS variant
    wgs_shared[['allele', 'total']] = wgs_shared[wgs_shared.columns[9]].str.split(':', expand=True).iloc[:, [1, 2]]
    wgs_shared[['ref', 'alt']] = wgs_shared['allele'].str.split(',', expand = True).iloc[:, [0, 1]]
    wgs_shared[['ref', 'alt', 'total']] = wgs_shared[['ref', 'alt', 'total']].astype(float)
    wgs_shared['vaf'] = wgs_shared['alt']/wgs_shared['total']
    wgs_shared['type'] = "wgs_shared"
    
    rna_shared['gene'] = rna_shared['INFO'].str.split('|').str[3]
    
    shared_dep = rna_shared
    shared2_dep = wgs_shared

    # merge and sum vaf by gene
    df = shared_dep.merge(shared2_dep, on=['#CHROM', 'POS', 'REF', 'ALT'], how='inner')
    df.head()
    
    vaf_df = df.groupby("gene")[["vaf_x", "vaf_y"]].sum()
    vaf_df = vaf_df.iloc[1:]
    return vaf_df

In [ ]:
# import sample names
df = pd.read_table("samples.txt").iloc[:-1]

df["sample_name"] = (
    df["plac_num"].astype("Int64").astype(str) + "_" 
    + df["sample"].astype(str)
)
df.head()

In [ ]:
samples = df['sample_name']
samples

In [ ]:
for sample in samples:
    df = get_vaf(sample)
    df.to_csv("correlation/" + sample + "_vaf_correlation.csv")